# Adivinhar a proxima palavra

In [41]:
import pandas as pd

In [42]:
import torch
import torch.nn as nn
import torch.optim as optim

In [43]:
import numpy as np

In [44]:
from string import punctuation

In [45]:
from sklearn.feature_extraction.text import CountVectorizer

In [46]:
from sklearn.model_selection import train_test_split

In [47]:
import random

In [48]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

In [49]:
df = pd.read_csv("d:/git/dados/nlp/imdb-reviews-pt-br.csv", encoding="utf-8")

In [50]:
df = df.drop(columns=["id", "text_en", "sentiment"])

In [51]:
df

,text_pt
0,"Mais uma vez, o Sr. Costner arrumou um filme p..."
1,Este é um exemplo do motivo pelo qual a maiori...
2,"Primeiro de tudo eu odeio esses raps imbecis, ..."
3,Nem mesmo os Beatles puderam escrever músicas ...
4,Filmes de fotos de latão não é uma palavra apr...
...,...
49454,"Como a média de votos era muito baixa, e o fat..."
49455,O enredo teve algumas reviravoltas infelizes e...
49456,Estou espantado com a forma como este filme e ...
49457,A Christmas Together realmente veio antes do m...


In [52]:
table = str.maketrans("", "", punctuation + "\u200b")

def limpar( texto ):
    texto_limpo = texto.lower().translate(table)
    return texto_limpo

In [53]:
df["text_clean"] = df["text_pt"].apply(limpar)

In [54]:
df["text_clean"][49455]

'o enredo teve algumas reviravoltas infelizes e inacreditáveis no entanto a química entre mel brooks e leslie ann warren foi excelente a percepção de que ela vem há apenas alguns momentos fornece uma abordagem filosófica pela qual qualquer pessoa poderia captar e abraçar a vida esse foi um dos vários momentos que foram maravilhosamente memoráveis'

In [55]:
MAXIMO_PALAVRAS_POR_TEXTO = 20
MAXIMO_FRASES = 1000
dicionario = {
    "<UNKNOWN>": 0,
    "<PAD>": 1
}
contador_palavras = len(dicionario.keys())
contador_palavras

2

In [56]:
lista_numeros = []
for texto in df["text_clean"][0:MAXIMO_FRASES]:
    numeros = []
    frase = texto.split(" ")
    for palavra in frase[0:MAXIMO_PALAVRAS_POR_TEXTO]:
        if palavra not in dicionario: 
            dicionario[palavra] = contador_palavras
            contador_palavras += 1
        numeros.append(dicionario.get(palavra, 0))
    lista_numeros.append(numeros)

In [57]:
dicionario_reverso = {}
for chave in dicionario.keys():
    valor = dicionario[chave]
    dicionario_reverso[valor] = chave

In [58]:
MAX_PALAVRAS = contador_palavras
MAX_PALAVRAS

4267

In [59]:
Y_palavra = []
X_frases = []
for lista_palavras in lista_numeros:
    for i in range(2, len(lista_palavras)):
        sentenca = lista_palavras[0 : i]
        if len(sentenca) >= 2:
            saida = lista_palavras[i]
            palavra_target = dicionario_reverso[saida]
            Y_palavra.append(palavra_target)
            X_frases.append(sentenca)
    

In [60]:
frase_teste = 4
print("Saida: ", Y_palavra[frase_teste])
print("Entrada: ", X_frases[frase_teste])
print("Frase com todos os numeros: ", lista_numeros[0])

Saida:  arrumou
Entrada:  [2, 3, 4, 5, 6, 7]
Frase com todos os numeros:  [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 2, 13, 14, 15, 5, 16, 17, 18, 19]


In [61]:
print("Frases utilizadas com base ==>", len(lista_numeros))
print("Frases geradas para treinamento ==>", len(X_frases))
print("Saidas para treinamento ==>", len(Y_palavra))

Frases utilizadas com base ==> 1000
Frases geradas para treinamento ==> 18000
Saidas para treinamento ==> 18000


In [62]:
encoder = LabelEncoder()

In [63]:
Y_labels = encoder.fit_transform(Y_palavra)
Y_labels

array([3930, 2704, 3559, ..., 2205, 3427, 2895], shape=(18000,))

In [64]:
print(Y_labels[:20])

[3930 2704 3559  948  355 3860 1723 2973 2585 2385 3674 1260 3143 2704
 2634  229 1025 3715 3860 1590]


In [65]:
MAX_CLASSES = len(encoder.classes_)
MAX_CLASSES

4087

In [66]:
Y = torch.tensor(Y_labels, dtype=torch.long)
Y.shape

torch.Size([18000])

In [67]:
# Identificar qual frase tem a maior quantidade de palavras
max_size = 0
for frase in X_frases:
    if len(frase) > max_size:
        max_size = len(frase)
max_size        

19

In [68]:
def padding(list_numbers, size, padding_value):
    number_pads = size - len(list_numbers)
    frase_padded = []
    for i in range(number_pads):
        frase_padded.append( padding_value )
    frase_padded.extend( list_numbers )
    return frase_padded


In [69]:
lista_padded = []
for frase in X_frases:
    lista_padded.append( padding(frase, max_size, dicionario["<PAD>"]) )

In [70]:
### 
# [1 , 1 , 1 , 1 , 12, 17, 19, 10]
# [12, 17, 19, 10, 34, 78, 23, 19]
# [1 , 1 , 1 , 1 , 1 , 12, 17, 19]

In [71]:
len(lista_padded[0])

19

In [72]:
X = torch.tensor(lista_padded, dtype=torch.long)
X

tensor([[   1,    1,    1,  ...,    1,    2,    3],
        [   1,    1,    1,  ...,    2,    3,    4],
        [   1,    1,    1,  ...,    3,    4,    5],
        ...,
        [   1,    1, 2522,  ...,   30,  874, 2795],
        [   1, 2522,  323,  ...,  874, 2795, 2432],
        [2522,  323, 1667,  ..., 2795, 2432,  101]])

In [73]:
print("X: ", X.dtype, X.shape, X.ndim)
print("Y: ", Y.dtype, Y.shape, Y.ndim)

X:  torch.int64 torch.Size([18000, 19]) 2
Y:  torch.int64 torch.Size([18000]) 1


In [74]:
X_treino, X_teste, Y_treino, Y_teste = train_test_split(X, Y, test_size=0.2, random_state=100)

In [75]:
Y_treino
# X_treino[0].sum()

tensor([   0, 1723, 1031,  ...,  199, 3656, 2694])

In [76]:
class ProcessadorTexto( nn.Module ):
    def __init__(self, embeddings_in, embedding_out, padding_idx, num_classes, hidden_size):
        super(ProcessadorTexto, self).__init__()

        self.embedding = nn.Embedding(num_embeddings = embeddings_in, 
                                      embedding_dim = embedding_out, 
                                      padding_idx = padding_idx)
        self.rnn = nn.RNN(
            input_size = embedding_out,
            hidden_size = hidden_size,
            batch_first = True
        )

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        embedded = self.embedding( x )
        rnn_out, rnn_hidden = self.rnn( embedded )
        ultimo_estado = rnn_hidden[-1]
        logits = self.fc(ultimo_estado)
        return logits

In [77]:
print("MAX_CLASSES: ", MAX_CLASSES)
print("MAX_PALAVRAS: ", MAX_PALAVRAS)

MAX_CLASSES:  4087
MAX_PALAVRAS:  4267


In [78]:
# modelo = nn.Linear(in_features = MAXIMO_PALAVRAS_POR_TEXTO - 1, out_features = MAX_CLASSES)
# modelo = nn.Sequential(
#     nn.Linear(in_features = MAX_PALAVRAS, out_features=1),
#     nn.Sigmoid()
# )
padding_idx = dicionario["<PAD>"]
modelo = ProcessadorTexto(embeddings_in=MAX_PALAVRAS, embedding_out=50,
                          padding_idx=padding_idx, num_classes=MAX_CLASSES, hidden_size=10)


In [79]:
criterio = nn.CrossEntropyLoss() # Cross Entropy Loss
otimizador = optim.Adam( modelo.parameters(), lr=0.01 )

In [80]:
for epoca in range(1, 200):
    Y_hat = modelo( X_treino )
    print(Y_hat.shape)
    loss = criterio( Y_hat, Y_treino )
    otimizador.zero_grad()
    loss.backward()
    otimizador.step()
    if epoca % 10 == 0:
        print(f"Epoca: {epoca}\tLoss:{loss}")

torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
Epoca: 10	Loss:7.768791675567627
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
Epoca: 20	Loss:6.889500141143799
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
Epoca: 30	Loss:6.217223167419434
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400, 4087])
torch.Size([14400

In [88]:
texto_completo = df["text_clean"][2000]
texto = " ".join(texto_completo.split(" ")[0:5])
print("Texto escolhido: ", texto)
print("Texto completo: ", texto_completo)

Texto escolhido:  eu acabei de ver este
Texto completo:  eu acabei de ver este filme e tudo o que posso dizer é onde estão os discos desses dias isto parece que teria sido um ótimo segundo recurso em uma unidade em 1977 talvez jogando com um daqueles filmes de joan collins mas seu único valor assistindo agora se você está sentindo nostalgia para os anos 70 um enredo bobo cheio de buracos mas que lembra a época em que foi feito interessante ver melanie griffith tão nova e anne lockhart é bastante atraente embora não muito de uma atriz na verdade não há muita atuação acontecendo nesse filme é uma espécie de aventura de duques de hazzard sem um som ou um carregador dodge de 1969 saltando sobre as coisas no bosque mas há um cometa mecrury saltando sobre um depósito de lixo neste


In [89]:
unknown_word = dicionario["<UNKNOWN>"]
texto_em_numeros = []
texto_em_token = texto.split(" ")
for token in texto_em_token: 
    texto_em_numeros.append(dicionario.get(token, unknown_word))
texto_em_numeros

[40, 1086, 30, 410, 20]

In [90]:
texto_em_numeros_padded = padding(texto_em_numeros, max_size, padding_idx)
texto_em_numeros_padded

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 40, 1086, 30, 410, 20]

In [91]:
X_predict = torch.tensor([texto_em_numeros_padded], dtype=torch.int32)
X_predict

tensor([[   1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,   40, 1086,   30,  410,   20]], dtype=torch.int32)

In [92]:
Y_predict = modelo(X_predict)
numero_palavra = np.argmax(Y_predict.detach().numpy())
encoder.inverse_transform([numero_palavra]) 

array(['filme'], dtype='<U20')